# Task 1.1
### Data Cleaning

This section loads the NSW region summary dataset and prepares a cleaned version for analysis. The main issue in this file is that many indicators are only available in selected years, so missing values should not be filled with zero. Instead, the workflow keeps the original values, standardises the column names, extracts the unit from each indicator description, counts how many valid years each indicator has, and reshapes the dataset into a long format for later statistics and visualisation.

In [197]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
data_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(data_path)
display(df.head())


,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 June,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 June,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 June,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 June,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 June,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


## Cleaning logic

The raw table is a wide table, which means each row is one indicator and each year is stored in a separate column. To make the data easier to analyse, the cleaning step below does three things. First, it standardises the text column names and year column names. Second, it separates the measurement unit from the description column and records how many non-null years each indicator has. Third, it reshapes the table into a long format so that each row represents one indicator-year value.

In [198]:
task1_df = df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
).copy()

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)
task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

task1_long_df = task1_df.melt(
    id_vars=["measure_code", "parent_description", "description_clean", "unit"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)
task1_analysis_df = (
    task1_long_df[task1_long_df["year"] != 2025]
    .drop_duplicates(subset=["measure_code", "description_clean", "year"])
    .copy()
)

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)

display(cleaning_summary_df)
display(task1_df.head())
display(task1_analysis_df.head())


,item,value
0,original_rows,800
1,rows_after_drop_all_null_years,800
2,long_table_rows,2874
3,analysis_rows_without_2025,2871
4,rows_with_only_one_non_null_year,105
5,non_null_values_in_2025,3


,measure_code,parent_description,description,y_2011,y_2015,y_2016,y_2017,y_2018,y_2019,y_2020,y_2021,y_2022,y_2023,y_2024,y_2025,unit,description_clean,non_null_years
0,ERP_P_20,Estimated resident population - year ended 30 June,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN,no.,Estimated resident population,6
1,ERP_21,Estimated resident population - year ended 30 June,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN,persons/km2,Population density,6
2,ERP_M_20,Estimated resident population - year ended 30 June,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN,no.,Estimated resident population - males,6
3,ERP_F_20,Estimated resident population - year ended 30 June,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN,no.,Estimated resident population - females,6
4,ERP_19,Estimated resident population - year ended 30 June,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN,years,Median age - males,6


,measure_code,parent_description,description_clean,unit,year,value
127,CENSUS_34,Aboriginal and Torres Strait Islander Peoples - Census,Aboriginal and Torres Strait Islander Peoples,no.,2011,172620.0
128,CENSUS_2,Aboriginal and Torres Strait Islander Peoples - Census,Aboriginal and Torres Strait Islander Peoples,%,2011,2.5
140,CENSUS_15,Religious affiliation - Census,Buddhism,%,2011,2.9
141,CENSUS_16,Religious affiliation - Census,Christianity,%,2011,64.5
142,CENSUS_17,Religious affiliation - Census,Hinduism,%,2011,1.7


# Derived Statistics 1.2

## 1.2.1 (by Jiamu Zheng)

### Statistic 1: Mean-to-median income ratio
Compares mean total income with median total income.

Raw data: `INCOME_36`, `INCOME_17`

Formula: `mean total income / median total income`

Interpretation: A value above 1 means mean income is higher than median income, which can suggest income inequality.


In [199]:


import pandas as pd

# Step 1: Identify income indicators related to total income
income_total_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "total income",
        case=False,
        na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(income_total_check)

# Step 2: Select the confirmed mean and median total income indicators
income_ratio_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin(["INCOME_17", "INCOME_36"])
][
    ["measure_code", "year", "value"]
].copy()

# Step 3: Reshape so each year has both median and mean income
income_ratio_wide = income_ratio_df.pivot(
    index="year",
    columns="measure_code",
    values="value"
).reset_index()

# Step 4: Rename columns based on the checked descriptions
income_ratio_wide = income_ratio_wide.rename(
    columns={
        "INCOME_17": "median_total_income",
        "INCOME_36": "mean_total_income"
    }
)

# Step 5: Calculate mean-to-median income ratio
income_ratio_wide["mean_to_median_ratio"] = (
    income_ratio_wide["mean_total_income"] /
    income_ratio_wide["median_total_income"]
)

display(income_ratio_wide)

,measure_code,parent_description,description_clean,unit
3638,INCOME_17,Personal income in Australia - year ended 30 June,Median total income (excl. Government pensions and allowances),$
3637,INCOME_18,Personal income in Australia - year ended 30 June,Total income (excl. Government pensions and allowances),$m
3635,INCOME_19,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances),no.
3636,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,years
3639,INCOME_36,Personal income in Australia - year ended 30 June,Mean total income (excl. Government pensions and allowances),$


measure_code,year,median_total_income,mean_total_income,mean_to_median_ratio
0,2018,47852.0,64528.0,1.348491
1,2019,49522.0,66310.0,1.339001
2,2020,50567.0,67677.0,1.338363
3,2021,53905.0,71919.0,1.334181
4,2022,55105.0,75511.0,1.370311


### Statistic 2: Income growth rate
Compares current-year median total income with previous-year median total income.

Raw data: `INCOME_17`

Formula: `(current median income - previous median income) / previous median income × 100`

Interpretation: Positive means median income increased; negative means median income decreased.


In [200]:
#Income growth rate
income_growth_df = task1_analysis_df[
    task1_analysis_df["measure_code"] == "INCOME_17"
].copy()


income_growth_df = income_growth_df[
    ["year", "value"]
].rename(
    columns={"value": "median_total_income"}
)


income_growth_df = income_growth_df.sort_values("year")


income_growth_df["income_growth_rate"] = (
    income_growth_df["median_total_income"]
    .pct_change() * 100
)

display(income_growth_df)

,year,median_total_income,income_growth_rate
3638,2018,47852.0,NaN
4438,2019,49522.0,3.489927
5238,2020,50567.0,2.110173
6038,2021,53905.0,6.601143
6838,2022,55105.0,2.226139


### Statistic 3: Investment income premium
Compares mean investment income with mean employee income.

Raw data: `INCOME_27`, `INCOME_21`

Formula: `mean investment income / mean employee income`

Interpretation: A higher value means investment income is larger relative to employee income.


In [201]:

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
# Step 1: Explore all mean income indicators
mean_income_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "Mean", case=False, na=False
    ) &
    task1_analysis_df["description_clean"].str.contains(
        "income", case=False, na=False
    ) &
    (task1_analysis_df["unit"] == "$")
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(mean_income_check)

# Step 2: Based on the table above, select:
# INCOME_21 = Mean employee income
# INCOME_27 = Mean investment income

investment_premium_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin(["INCOME_21", "INCOME_27"])
][
    ["measure_code", "year", "value"]
].copy()

# Step 3: Reshape so each year has both income types
investment_premium_wide = investment_premium_df.pivot(
    index="year",
    columns="measure_code",
    values="value"
).reset_index()

# Step 4: Rename columns based on the identified indicators
investment_premium_wide = investment_premium_wide.rename(columns={
    "INCOME_21": "mean_employee_income",
    "INCOME_27": "mean_investment_income"
})

# Step 5: Calculate investment income premium
investment_premium_wide["investment_income_premium"] = (
    investment_premium_wide["mean_investment_income"] /
    investment_premium_wide["mean_employee_income"]
)

display(investment_premium_wide)

,measure_code,parent_description,description_clean,unit
3615,INCOME_21,Personal income in Australia - year ended 30 June,Mean employee income,$
3621,INCOME_24,Personal income in Australia - year ended 30 June,Mean own unincorporated business income,$
3627,INCOME_27,Personal income in Australia - year ended 30 June,Mean investment income,$
3633,INCOME_30,Personal income in Australia - year ended 30 June,Mean superannuation and annuity income,$
3639,INCOME_36,Personal income in Australia - year ended 30 June,Mean total income (excl. Government pensions and allowances),$


measure_code,year,mean_employee_income,mean_investment_income,investment_income_premium
0,2018,64509.0,9525.0,0.147654
1,2019,66377.0,9698.0,0.146105
2,2020,68690.0,9373.0,0.136454
3,2021,71848.0,10133.0,0.141034
4,2022,74232.0,12229.0,0.164740


### Statistic 4: High-income share vs low-income share
Compares high-income population share with low-income population share.

Raw data: `PERSINC_5`, `PERSINC_6`, `PERSINC_2`, `PERSINC_7`

Formula: `high-income share / low-income share`

Interpretation: A higher value means the high-income group is larger relative to the low-income group.


In [202]:


# Step 1: Explore personal income bracket indicators
personal_income_check = task1_analysis_df[
    task1_analysis_df["parent_description"].str.contains(
        "Total personal income", case=False, na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(personal_income_check)

# Step 2: Select low-income and high-income brackets
income_share_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin([
        "PERSINC_2",  # $1-$499 per week
        "PERSINC_5",  # $2000-$2999 per week
        "PERSINC_6",  # $3000 or more per week
        "PERSINC_7"   # Nil income
    ])
].copy()

# Step 3: Classify selected indicators into low-income and high-income groups
income_share_df["income_group"] = income_share_df["measure_code"].map({
    "PERSINC_2": "low_income",
    "PERSINC_7": "low_income",
    "PERSINC_5": "high_income",
    "PERSINC_6": "high_income"
})

# Step 4: Sum the percentages within each income group by year
income_share_summary = (
    income_share_df
    .groupby(["year", "income_group"])["value"]
    .sum()
    .reset_index()
)

# Step 5: Reshape so each year has both low-income and high-income shares
income_share_wide = income_share_summary.pivot(
    index="year",
    columns="income_group",
    values="value"
).reset_index()

# Step 6: Calculate high-to-low income ratio
income_share_wide["high_to_low_income_ratio"] = (
    income_share_wide["high_income"] /
    income_share_wide["low_income"]
)

display(income_share_wide)

,measure_code,parent_description,description_clean,unit
2329,INCMIG_2,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$1-$499 per week,%
2330,INCMIG_3,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$500-$999 per week,%
2331,INCMIG_4,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$1000-$1999 per week,%
2332,INCMIG_5,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$2000-$2999 per week,%
2333,INCMIG_6,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$3000 or more per week,%
2334,INCMIG_7,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Nil income,%
2335,INCMIG_8,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Negative income,%
2336,INCMIG_9,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Income inadequately described or not stated,%
1991,PERSINC_2,Total personal income (weekly) - Persons aged 15 years and over - Census,$1-$499 per week,%
1992,PERSINC_3,Total personal income (weekly) - Persons aged 15 years and over - Census,$500-$999 per week,%


income_group,year,high_income,low_income,high_to_low_income_ratio
0,2016,8.8,37.0,0.237838
1,2021,13.6,31.2,0.435897


### Statistic 5: Employee income as main source of income
Shows the share of people whose main source of income is employee income.

Raw data: `INCOME_22`

Formula: selected percentage value from `INCOME_22`

Interpretation: A higher value means more people mainly rely on employee income.


In [203]:

# Step 1: Explore income source indicators
income_source_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "main source of income", case=False, na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(income_source_check)

# Step 2: Select employee income as main source of income
employee_income_source_df = task1_analysis_df[
    task1_analysis_df["measure_code"] == "INCOME_22"
][
    ["year", "value"]
].rename(
    columns={"value": "employee_income_main_source_share"}
).copy()

employee_income_source_df = employee_income_source_df.sort_values("year")

display(employee_income_source_df)

,measure_code,parent_description,description_clean,unit
3616,INCOME_22,Personal income in Australia - year ended 30 June,Employee income as main source of income,%
3622,INCOME_25,Personal income in Australia - year ended 30 June,Own unincorporated business income as main source of income,%
3628,INCOME_28,Personal income in Australia - year ended 30 June,Investment income as main source of income,%
3634,INCOME_31,Personal income in Australia - year ended 30 June,Superannuation and annuity income as main source of income,%


,year,employee_income_main_source_share
3616,2018,79.8
4416,2019,79.7
5216,2020,79.7
6016,2021,80.0
6816,2022,80.4


## 1.2.2 (by Feiyan Lan)

### Age-related derived statistics
This section creates five derived age statistics from the cleaned Task 1 data.


In [204]:
# Prepare age-related data

import pandas as pd
import numpy as np

# Step 1: Filter rows related to median age
age_df = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains("median age", case=False, na=False)
].copy()

# Step 2: Check which age indicators are included
age_check = age_df[
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates()

display(age_check)

# Step 3: Check available years for age data
available_age_years = sorted(age_df["year"].dropna().unique())

print("Available years for age data:")
print(available_age_years)

# Step 4: Set start year and latest available year
start_year = 2019
end_year = max(available_age_years)

print("Start year:", start_year)
print("End year:", end_year)

,measure_code,parent_description,description_clean,unit
160,ING_MA_M,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - males,years
161,ING_MA_F,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - females,years
162,ING_MA_P,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - persons,years
3612,INCOME_20,Personal income in Australia - year ended 30 June,Employee income earners - median age,years
3618,INCOME_23,Personal income in Australia - year ended 30 June,Own unincorporated business income earners - median age,years
3624,INCOME_26,Personal income in Australia - year ended 30 June,Investment income earners - median age,years
3630,INCOME_29,Personal income in Australia - year ended 30 June,Superannuation and annuity income earners - median age,years
3636,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,years
4004,ERP_19,Estimated resident population - year ended 30 June,Median age - males,years
4005,ERP_22,Estimated resident population - year ended 30 June,Median age - females,years


Available years for age data:
[np.int64(2011), np.int64(2016), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Start year: 2019
End year: 2024


### Statistic 1: Median age change
Compares median age in 2019 with the latest available year.

Raw data: median age indicators from `age_df`

Formula: `latest available median age - 2019 median age`

Interpretation: Positive means median age increased over time.


In [205]:
# Statistic 1: Median Age Change

# Step 1: Convert age data from long format to wide format
# Each row is one median age indicator, and each year becomes one column
age_wide = age_df.pivot_table(
    index=["measure_code", "parent_description", "description_clean", "unit"],
    columns="year",
    values="value",
    aggfunc="first"
).reset_index()

# Step 2: Calculate the change in median age
# Formula: latest year median age - 2019 median age
age_wide["median_age_change"] = age_wide[end_year] - age_wide[start_year]

# Step 3: Select useful columns for display
median_age_change = age_wide[
    [
        "measure_code",
        "parent_description",
        "description_clean",
        start_year,
        end_year,
        "median_age_change"
    ]
]

# Step 4: Show the result
display(median_age_change)

year,measure_code,parent_description,description_clean,2019,2024,median_age_change
0,ERP_19,Estimated resident population - year ended 30 June,Median age - males,36.8,37.5,0.7
1,ERP_22,Estimated resident population - year ended 30 June,Median age - females,38.6,39.4,0.8
2,ERP_23,Estimated resident population - year ended 30 June,Median age - persons,37.7,38.4,0.7
3,INCOME_20,Personal income in Australia - year ended 30 June,Employee income earners - median age,38.0,NaN,NaN
4,INCOME_23,Personal income in Australia - year ended 30 June,Own unincorporated business income earners - median age,46.0,NaN,NaN
5,INCOME_26,Personal income in Australia - year ended 30 June,Investment income earners - median age,44.0,NaN,NaN
6,INCOME_29,Personal income in Australia - year ended 30 June,Superannuation and annuity income earners - median age,59.0,NaN,NaN
7,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,41.0,NaN,NaN
8,ING_MA_F,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - females,NaN,NaN,NaN
9,ING_MA_M,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - males,NaN,NaN,NaN


### Statistic 2: Median age growth rate
Compares median age growth from 2019 to the latest available year.

Raw data: median age indicators from `age_df`

Formula: `(latest median age - 2019 median age) / 2019 median age × 100`

Interpretation: A higher value means median age grew faster.


In [206]:
# Statistic 2: Median Age Growth Rate

# Step 1: Create a copy of the wide age table
age_growth_df = age_wide.copy()

# Step 2: Calculate the percentage growth rate
# Formula: ((latest year value - 2019 value) / 2019 value) * 100
age_growth_df["median_age_growth_rate"] = (
    (age_growth_df[end_year] - age_growth_df[start_year])
    / age_growth_df[start_year]
    * 100
)

# Step 3: Select useful columns for display
median_age_growth_rate = age_growth_df[
    [
        "measure_code",
        "parent_description",
        "description_clean",
        start_year,
        end_year,
        "median_age_growth_rate"
    ]
]

# Step 4: Show the result
display(median_age_growth_rate)

year,measure_code,parent_description,description_clean,2019,2024,median_age_growth_rate
0,ERP_19,Estimated resident population - year ended 30 June,Median age - males,36.8,37.5,1.902174
1,ERP_22,Estimated resident population - year ended 30 June,Median age - females,38.6,39.4,2.072539
2,ERP_23,Estimated resident population - year ended 30 June,Median age - persons,37.7,38.4,1.856764
3,INCOME_20,Personal income in Australia - year ended 30 June,Employee income earners - median age,38.0,NaN,NaN
4,INCOME_23,Personal income in Australia - year ended 30 June,Own unincorporated business income earners - median age,46.0,NaN,NaN
5,INCOME_26,Personal income in Australia - year ended 30 June,Investment income earners - median age,44.0,NaN,NaN
6,INCOME_29,Personal income in Australia - year ended 30 June,Superannuation and annuity income earners - median age,59.0,NaN,NaN
7,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,41.0,NaN,NaN
8,ING_MA_F,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - females,NaN,NaN,NaN
9,ING_MA_M,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - males,NaN,NaN,NaN


### Statistic 3: Male vs female median age difference
Compares female median age with male median age in the latest available year.

Raw data: male and female median age rows from `age_wide`

Formula: `female median age - male median age`

Interpretation: Positive means female median age is higher than male median age.


In [207]:
# Statistic 3: Male vs Female Median Age Difference

# Step 1: Find the male median age row
# Important: use regex so that "males" does not accidentally match "females"
male_median_age_row = age_wide[
    age_wide["description_clean"].str.contains(r"\bmales\b", case=False, na=False, regex=True)
]

# Step 2: Find the female median age row
female_median_age_row = age_wide[
    age_wide["description_clean"].str.contains(r"\bfemales\b", case=False, na=False, regex=True)
]

# Step 3: Extract the latest available values
male_median_age = male_median_age_row[end_year].iloc[0]
female_median_age = female_median_age_row[end_year].iloc[0]

# Step 4: Calculate the difference
# Formula: female median age - male median age
median_age_gender_difference = female_median_age - male_median_age

# Step 5: Store result in a small DataFrame
gender_age_difference = pd.DataFrame({
    "statistic": ["Female median age - Male median age"],
    "year": [end_year],
    "male_median_age": [male_median_age],
    "female_median_age": [female_median_age],
    "difference": [median_age_gender_difference]
})

# Step 6: Show the result
display(gender_age_difference)

,statistic,year,male_median_age,female_median_age,difference
0,Female median age - Male median age,2024,37.5,39.4,1.9


### Statistic 4: Highest median age year
Finds the year when each median age indicator reached its highest value.

Raw data: median age indicators from `age_df`

Formula: year with maximum median age value

Interpretation: Shows when each age indicator peaked.


In [208]:
# Statistic 4: Highest Median Age Year

# Step 1: Create a copy of the age dataset
highest_age_df = age_df.copy()

# Step 2: Sort values from highest to lowest
# Then group by each median age indicator and keep the first row
highest_median_age_year = (
    highest_age_df
    .sort_values("value", ascending=False)
    .groupby(["measure_code", "parent_description", "description_clean", "unit"])
    .first()
    .reset_index()
)

# Step 3: Rename columns for clarity
highest_median_age_year = highest_median_age_year.rename(
    columns={
        "year": "highest_median_age_year",
        "value": "highest_median_age_value"
    }
)

# Step 4: Select useful columns for display
highest_median_age_year = highest_median_age_year[
    [
        "measure_code",
        "parent_description",
        "description_clean",
        "highest_median_age_year",
        "highest_median_age_value"
    ]
]

# Step 5: Show the result
display(highest_median_age_year)

,measure_code,parent_description,description_clean,highest_median_age_year,highest_median_age_value
0,ERP_19,Estimated resident population - year ended 30 June,Median age - males,2021,37.7
1,ERP_22,Estimated resident population - year ended 30 June,Median age - females,2022,39.6
2,ERP_23,Estimated resident population - year ended 30 June,Median age - persons,2022,38.7
3,INCOME_20,Personal income in Australia - year ended 30 June,Employee income earners - median age,2021,39.0
4,INCOME_23,Personal income in Australia - year ended 30 June,Own unincorporated business income earners - median age,2018,47.0
5,INCOME_26,Personal income in Australia - year ended 30 June,Investment income earners - median age,2021,45.0
6,INCOME_29,Personal income in Australia - year ended 30 June,Superannuation and annuity income earners - median age,2021,60.0
7,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,2018,41.0
8,ING_MA_F,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - females,2021,24.3
9,ING_MA_M,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - males,2021,22.6


### Statistic 5: Median age range
Compares the highest and lowest median age values across available years.

Raw data: median age indicators from `age_df`

Formula: `maximum median age - minimum median age`

Interpretation: A larger range means median age changed more across the period.


In [209]:
# Statistic 5: Median Age Range

# Step 1: Group by each median age indicator
# Then calculate the lowest and highest median age values
age_range_df = (
    age_df
    .groupby(["measure_code", "parent_description", "description_clean", "unit"])
    .agg(
        lowest_median_age_value=("value", "min"),
        highest_median_age_value=("value", "max")
    )
    .reset_index()
)

# Step 2: Calculate the range
# Formula: maximum median age - minimum median age
age_range_df["median_age_range"] = (
    age_range_df["highest_median_age_value"]
    - age_range_df["lowest_median_age_value"]
)

# Step 3: Select useful columns for display
median_age_range = age_range_df[
    [
        "measure_code",
        "parent_description",
        "description_clean",
        "lowest_median_age_value",
        "highest_median_age_value",
        "median_age_range"
    ]
]

# Step 4: Show the result
display(median_age_range)

,measure_code,parent_description,description_clean,lowest_median_age_value,highest_median_age_value,median_age_range
0,ERP_19,Estimated resident population - year ended 30 June,Median age - males,36.8,37.7,0.9
1,ERP_22,Estimated resident population - year ended 30 June,Median age - females,38.6,39.6,1.0
2,ERP_23,Estimated resident population - year ended 30 June,Median age - persons,37.7,38.7,1.0
3,INCOME_20,Personal income in Australia - year ended 30 June,Employee income earners - median age,38.0,39.0,1.0
4,INCOME_23,Personal income in Australia - year ended 30 June,Own unincorporated business income earners - median age,46.0,47.0,1.0
5,INCOME_26,Personal income in Australia - year ended 30 June,Investment income earners - median age,44.0,45.0,1.0
6,INCOME_29,Personal income in Australia - year ended 30 June,Superannuation and annuity income earners - median age,59.0,60.0,1.0
7,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pensions and allowances) - median age,41.0,41.0,0.0
8,ING_MA_F,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - females,22.2,24.3,2.1
9,ING_MA_M,Estimated resident Aboriginal and Torres Strait Islander population - at 30 June,Median age - males,20.6,22.6,2.0


## 1.2.3 (by Qingyi Zhu)


### Population-related derived statistics
This section creates five derived population statistics from cleaned Task 1 data.


In [210]:
population_codes = {
    "ERP_P_20": "total_population",
    "ERP_21": "population_density",
    "ERP_M_20": "male_population",
    "ERP_F_20": "female_population",
    "ERP_18": "working_age_population",
}

population_base_wide = (
    task1_analysis_df.loc[
        task1_analysis_df["measure_code"].isin(population_codes),
        ["year", "measure_code", "value"],
    ]
    .pivot(index="year", columns="measure_code", values="value")
    .rename(columns=population_codes)
    .reset_index()
    .sort_values("year")
)
population_base_wide.columns.name = None

display(population_base_wide)


,year,working_age_population,population_density,female_population,male_population,total_population
0,2019,5240576.0,10.0,4047296.0,3999452.0,8046748.0
1,2020,5255515.0,10.1,4079900.0,4030710.0,8110610.0
2,2021,5209700.0,10.1,4071669.0,4025393.0,8097062.0
3,2022,5247219.0,10.2,4106941.0,4059763.0,8166704.0
4,2023,5388091.0,10.4,4192167.0,4149032.0,8341199.0
5,2024,5490199.0,10.6,4261453.0,4217861.0,8479314.0


### Statistic 1: Total population growth rate
Compares current-year total population with previous-year total population.

Raw data: `ERP_P_20`

Formula: `(current population - previous population) / previous population × 100`

Interpretation: Positive means population increased; negative means population decreased.


In [211]:
population_growth_df = population_base_wide[["year", "total_population"]].copy()
population_growth_df["annual_population_change"] = population_growth_df["total_population"].diff()
population_growth_df["population_growth_rate"] = population_growth_df["total_population"].pct_change() * 100

display(population_growth_df)


,year,total_population,annual_population_change,population_growth_rate
0,2019,8046748.0,NaN,NaN
1,2020,8110610.0,63862.0,0.793637
2,2021,8097062.0,-13548.0,-0.167040
3,2022,8166704.0,69642.0,0.860090
4,2023,8341199.0,174495.0,2.136664
5,2024,8479314.0,138115.0,1.655817


### Statistic 2: Population density and implied area
Combines total population with population density to estimate implied land area.

Raw data: `ERP_P_20`, `ERP_21`

Formula: `implied_area = total population / population density`

Interpretation: A stable implied area suggests the population and density indicators are consistent.


In [212]:
population_density_area_df = population_base_wide[[
    "year", "total_population", "population_density"
]].copy()
population_density_area_df["implied_area_sqkm"] = (
    population_density_area_df["total_population"] / population_density_area_df["population_density"]
)

display(population_density_area_df)


,year,total_population,population_density,implied_area_sqkm
0,2019,8046748.0,10.0,804674.800000
1,2020,8110610.0,10.1,803030.693069
2,2021,8097062.0,10.1,801689.306931
3,2022,8166704.0,10.2,800657.254902
4,2023,8341199.0,10.4,802038.365385
5,2024,8479314.0,10.6,799935.283019


### Statistic 3: Male-to-female population ratio
Compares male population with female population.

Raw data: `ERP_M_20`, `ERP_F_20`

Formula: `male population / female population × 100`

Interpretation: A value close to 100 means the male and female populations are balanced.


In [213]:
gender_population_df = population_base_wide[[
    "year", "male_population", "female_population"
]].copy()
gender_population_df["males_per_100_females"] = (
    gender_population_df["male_population"] / gender_population_df["female_population"] * 100
)

display(gender_population_df)


,year,male_population,female_population,males_per_100_females
0,2019,3999452.0,4047296.0,98.817877
1,2020,4030710.0,4079900.0,98.794333
2,2021,4025393.0,4071669.0,98.863464
3,2022,4059763.0,4106941.0,98.851262
4,2023,4149032.0,4192167.0,98.971057
5,2024,4217861.0,4261453.0,98.977063


### Statistic 4: Working-age dependency ratio
Compares non-working-age population with working-age population.

Raw data: `ERP_P_20`, `ERP_18`

Formula: `(total population - working-age population) / working-age population × 100`

Interpretation: A higher value means more non-working-age people per 100 working-age people.


In [214]:
dependency_ratio_df = population_base_wide[[
    "year", "total_population", "working_age_population"
]].copy()
dependency_ratio_df["non_working_age_population"] = (
    dependency_ratio_df["total_population"] - dependency_ratio_df["working_age_population"]
)
dependency_ratio_df["dependency_ratio"] = (
    dependency_ratio_df["non_working_age_population"]
    / dependency_ratio_df["working_age_population"]
    * 100
)

display(dependency_ratio_df)


,year,total_population,working_age_population,non_working_age_population,dependency_ratio
0,2019,8046748.0,5240576.0,2806172.0,53.547015
1,2020,8110610.0,5255515.0,2855095.0,54.325694
2,2021,8097062.0,5209700.0,2887362.0,55.422807
3,2022,8166704.0,5247219.0,2919485.0,55.638711
4,2023,8341199.0,5388091.0,2953108.0,54.808057
5,2024,8479314.0,5490199.0,2989115.0,54.444566


### Statistic 5: Ageing index
Compares seniors aged 65+ with children aged 0-14.

Raw data: `ERP_P_15`-`ERP_P_19`, `ERP_P_2`-`ERP_P_4`

Formula: `seniors 65+ / children 0-14 × 100`

Interpretation: A higher value means the population structure is older.


In [215]:
children_codes = ["ERP_P_2", "ERP_P_3", "ERP_P_4"]
senior_codes = ["ERP_P_15", "ERP_P_16", "ERP_P_17", "ERP_P_18", "ERP_P_19"]

children = (
    task1_analysis_df[task1_analysis_df["measure_code"].isin(children_codes)]
    .groupby("year")["value"]
    .sum()
)
seniors = (
    task1_analysis_df[task1_analysis_df["measure_code"].isin(senior_codes)]
    .groupby("year")["value"]
    .sum()
)

ageing_index_df = pd.DataFrame({
    "children_0_14": children,
    "seniors_65_plus": seniors,
}).reset_index()
ageing_index_df["ageing_index"] = (
    ageing_index_df["seniors_65_plus"] / ageing_index_df["children_0_14"] * 100
)

display(ageing_index_df)


,year,children_0_14,seniors_65_plus,ageing_index
0,2019,1491711.0,1314461.0,88.117672
1,2020,1495841.0,1359254.0,90.868882
2,2021,1493689.0,1393673.0,93.304095
3,2022,1494960.0,1424525.0,95.288503
4,2023,1493417.0,1459691.0,97.741689
5,2024,1489028.0,1500087.0,100.742699


## 1.2.4 Housing-related derived statistics
This section creates five derived housing statistics from the cleaned Task 1 data.


In [216]:
vehicle_groups = {
    "VEHIC_2": "no_motor_vehicle",
    "VEHIC_3": "one_motor_vehicle",
    "VEHIC_4": "two_motor_vehicles",
    "VEHIC_5": "three_motor_vehicles",
    "VEHIC_6": "four_or_more_motor_vehicles",
}

vehicle_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin(vehicle_groups)
][["measure_code", "year", "value"]].copy()
vehicle_df["vehicle_group"] = vehicle_df["measure_code"].map(vehicle_groups)

vehicle_wide = (
    vehicle_df
    .pivot(index="year", columns="vehicle_group", values="value")
    .reset_index()
    .sort_values("year")
)
vehicle_wide.columns.name = None
vehicle_wide["total_private_households"] = vehicle_wide[list(vehicle_groups.values())].sum(axis=1)

display(vehicle_wide)


,year,four_or_more_motor_vehicles,no_motor_vehicle,one_motor_vehicle,three_motor_vehicles,two_motor_vehicles,total_private_households
0,2011,115060.0,258153.0,933951.0,245022.0,840660.0,2392846.0
1,2016,152006.0,239627.0,946160.0,283045.0,887849.0,2508687.0
2,2021,187380.0,262031.0,1096761.0,321310.0,989258.0,2856740.0


### Statistic 1: Total private households growth
Compares total private households across Census years.

Raw data: `VEHIC_2`-`VEHIC_6`

Formula: `(current total households - previous total households) / previous total households × 100`

Interpretation: A positive value means the number of private households increased.


In [217]:
household_growth_df = vehicle_wide[["year", "total_private_households"]].copy()
household_growth_df["household_change"] = household_growth_df["total_private_households"].diff()
household_growth_df["household_growth_rate"] = (
    household_growth_df["total_private_households"].pct_change() * 100
)

display(household_growth_df)


,year,total_private_households,household_change,household_growth_rate
0,2011,2392846.0,NaN,NaN
1,2016,2508687.0,115841.0,4.841139
2,2021,2856740.0,348053.0,13.873911


### Statistic 2: Households with no motor vehicle
Compares households with no motor vehicle with total private households.

Raw data: `VEHIC_2`, `VEHIC_2`-`VEHIC_6`

Formula: `no motor vehicle households / total private households × 100`

Interpretation: A higher value means a larger share of households do not have a motor vehicle.


In [218]:
no_vehicle_share_df = vehicle_wide[[
    "year", "no_motor_vehicle", "total_private_households"
]].copy()
no_vehicle_share_df["no_vehicle_share"] = (
    no_vehicle_share_df["no_motor_vehicle"]
    / no_vehicle_share_df["total_private_households"]
    * 100
)

display(no_vehicle_share_df)


,year,no_motor_vehicle,total_private_households,no_vehicle_share
0,2011,258153.0,2392846.0,10.788534
1,2016,239627.0,2508687.0,9.551889
2,2021,262031.0,2856740.0,9.172378


### Statistic 3: Average motor vehicles per household
Combines household counts by vehicle category to estimate average vehicle ownership.

Raw data: `VEHIC_2`-`VEHIC_6`

Formula: `weighted motor vehicle count / total private households`

Interpretation: A higher value means households own more motor vehicles on average.


In [219]:
vehicle_weights = {
    "no_motor_vehicle": 0,
    "one_motor_vehicle": 1,
    "two_motor_vehicles": 2,
    "three_motor_vehicles": 3,
    "four_or_more_motor_vehicles": 4,
}

average_vehicle_df = vehicle_wide[["year", "total_private_households"]].copy()
average_vehicle_df["weighted_vehicle_count"] = sum(
    vehicle_wide[col] * weight for col, weight in vehicle_weights.items()
)
average_vehicle_df["average_vehicles_per_household"] = (
    average_vehicle_df["weighted_vehicle_count"]
    / average_vehicle_df["total_private_households"]
)

display(average_vehicle_df)


,year,total_private_households,weighted_vehicle_count,average_vehicles_per_household
0,2011,2392846.0,3810577.0,1.592487
1,2016,2508687.0,4179017.0,1.665818
2,2021,2856740.0,4788727.0,1.676291


### Statistic 4: Urban intensive land use proportion
Compares urban intensive land use with total land use.

Raw data: `LANDUSE_13`, `LANDUSE_19`

Formula: `urban intensive land use / total land use × 100`

Interpretation: A higher value means a larger share of land is used for intensive urban purposes.


In [220]:
land_use_wide = (
    task1_analysis_df[task1_analysis_df["measure_code"].isin(["LANDUSE_13", "LANDUSE_19"])]
    .pivot(index="year", columns="measure_code", values="value")
    .rename(columns={"LANDUSE_13": "urban_intensive_use", "LANDUSE_19": "total_land_use"})
    .reset_index()
)
land_use_wide.columns.name = None

urban_land_use_df = land_use_wide[["year", "urban_intensive_use", "total_land_use"]].copy()
urban_land_use_df["urban_intensive_land_use_share"] = (
    urban_land_use_df["urban_intensive_use"]
    / urban_land_use_df["total_land_use"]
    * 100
)

display(urban_land_use_df)


,year,urban_intensive_use,total_land_use,urban_intensive_land_use_share
0,2016,684281.0,80129856.0,0.853965


### Statistic 5: Population density growth
Compares current-year population density with previous-year population density.

Raw data: `ERP_21`

Formula: `(current density - previous density) / previous density × 100`

Interpretation: A positive value means population density increased.


In [221]:
housing_density_growth_df = (
    task1_analysis_df[task1_analysis_df["measure_code"] == "ERP_21"]
    [["year", "value"]]
    .rename(columns={"value": "population_density"})
    .sort_values("year")
    .reset_index(drop=True)
)
housing_density_growth_df["density_change"] = housing_density_growth_df["population_density"].diff()
housing_density_growth_df["density_growth_rate"] = (
    housing_density_growth_df["population_density"].pct_change() * 100
)

display(housing_density_growth_df)


,year,population_density,density_change,density_growth_rate
0,2019,10.0,NaN,NaN
1,2020,10.1,0.1,1.000000
2,2021,10.1,0.0,0.000000
3,2022,10.2,0.1,0.990099
4,2023,10.4,0.2,1.960784
5,2024,10.6,0.2,1.923077


# Task 2

This section builds the geographic dataset for four selected areas. Each area is linked to one SA4 zone. For each selected SA4, the workflow uses the local ABS 2021 SA4 and SA2 shapefiles to identify the SA2 regions contained inside that SA4. It then uses the NSW POI API to collect POIs around each SA2 bounding box and filters the returned points with the SA2 polygon before saving the combined results to a local SQLite database.

## Selected SA4 codes

The group uses four SA4 codes: `120`, `117`, `118`, and `125`. The setup cell below matches each code to its official SA4 name and defines a small helper function so the same logic can be reused for each selected area.

In [222]:
import importlib
import task2_sa2

importlib.reload(task2_sa2)

available_sa4s = task2_sa2.list_nsw_sa4s()
area_zone_df = pd.DataFrame(
    {
        "area_name": ["area_1", "area_2", "area_3", "area_4"],
        "sa4_code": ["120", "117", "118", "125"],
    }
)

area_zone_df = area_zone_df.merge(
    available_sa4s.rename(columns={"SA4_CODE": "sa4_code", "SA4_NAME": "sa4_name"}),
    on="sa4_code",
    how="left",
)

def run_area_zone(area_name, sa4_code):
    sa2_bbox_df = task2_sa2.get_sa2_bbox_df_by_code(sa4_code)
    poi_df = task2_sa2.get_sa4_poi_df_by_code(sa4_code)
    sa2_bbox_df["area_name"] = area_name
    poi_df["area_name"] = area_name
    poi_count_df = poi_df.groupby("sa2_name").size().reset_index(name="poi_count")
    return sa2_bbox_df, poi_df, poi_count_df.sort_values("poi_count", ascending=False)

display(area_zone_df)


,area_name,sa4_code,sa4_name
0,area_1,120,Sydney - Inner West
1,area_2,117,Sydney - City and Inner South
2,area_3,118,Sydney - Eastern Suburbs
3,area_4,125,Sydney - Parramatta


## Area 1

Area 1 is linked to `SA4_CODE = 120`, which corresponds to `Sydney - Inner West`. The cell below returns three tables: the SA2 regions contained in this SA4, a sample of the POI rows collected for this zone, and a count of how many POIs were assigned to each SA2.

In [223]:
area_1_sa2_bbox_df, area_1_poi_df, area_1_poi_count_df = run_area_zone("area_1", "120")
display(area_1_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_1_poi_df.head())
display(area_1_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_1,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita
1,area_1,120,Sydney - Inner West,120011385,Drummoyne - Rodd Point
2,area_1,120,Sydney - Inner West,120011386,Five Dock - Abbotsford
3,area_1,120,Sydney - Inner West,120011672,Concord West - North Strathfield
4,area_1,120,Sydney - Inner West,120011673,Rhodes
5,area_1,120,Sydney - Inner West,120021387,Balmain
6,area_1,120,Sydney - Inner West,120021389,Lilyfield - Rozelle
7,area_1,120,Sydney - Inner West,120021674,Annandale (NSW)
8,area_1,120,Sydney - Inner West,120021675,Leichhardt
9,area_1,120,Sydney - Inner West,120031392,Canterbury (North) - Ashbury


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,849,None,Place Of Worship,1,151.109531,-33.860946,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
1,852,None,Place Of Worship,1,151.100483,-33.861704,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
2,1120,CONCORD GOLF COURSE,Golf Course,3,151.097332,-33.850397,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
3,1121,PRINCE EDWARD PARK,Park,3,151.119009,-33.852815,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
4,1122,MASSEY PARK GOLF COURSE,Golf Course,3,151.111647,-33.853167,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1


,sa2_name,poi_count
10,Drummoyne - Rodd Point,267
16,Lilyfield - Rozelle,235
15,Leichhardt,192
3,Balmain,152
13,Haberfield - Summer Hill,121
6,Concord - Mortlake - Cabarita,113
12,Five Dock - Abbotsford,89
4,Burwood (NSW),87
9,Croydon Park - Enfield,79
14,Homebush,63


## Area 2

Area 2 is linked to `SA4_CODE = 117`, which corresponds to `Sydney - City and Inner South`. The same process is repeated here so that the results remain separate by area.

In [224]:
area_2_sa2_bbox_df, area_2_poi_df, area_2_poi_count_df = run_area_zone("area_2", "117")
display(area_2_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_2_poi_df.head())
display(area_2_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_2,117,Sydney - City and Inner South,117011320,Banksmeadow
1,area_2,117,Sydney - City and Inner South,117011321,Botany
2,area_2,117,Sydney - City and Inner South,117011323,Pagewood - Hillsdale - Daceyville
3,area_2,117,Sydney - City and Inner South,117011324,Port Botany Industrial
4,area_2,117,Sydney - City and Inner South,117011325,Sydney Airport
5,area_2,117,Sydney - City and Inner South,117011634,Eastlakes
6,area_2,117,Sydney - City and Inner South,117011635,Mascot
7,area_2,117,Sydney - City and Inner South,117021327,Petersham - Stanmore
8,area_2,117,Sydney - City and Inner South,117021328,Sydenham - Tempe - St Peters
9,area_2,117,Sydney - City and Inner South,117021636,Marrickville - North


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,53133,BANKSMEADOW,Suburb,8,151.214736,-33.961926,117,Sydney - City and Inner South,117011320,Banksmeadow,area_2
1,97795,EAST BOTANY,Urban Place,8,151.215473,-33.946782,117,Sydney - City and Inner South,117011320,Banksmeadow,area_2
2,128047,None,Wharf,4,151.220483,-33.968008,117,Sydney - City and Inner South,117011320,Banksmeadow,area_2
3,128049,None,Wharf,4,151.220343,-33.967707,117,Sydney - City and Inner South,117011320,Banksmeadow,area_2
4,2999,ST MATTHEW'S ANGLICAN CHURCH,Place Of Worship,1,151.195602,-33.941055,117,Sydney - City and Inner South,117011321,Botany,area_2


,sa2_name,poi_count
20,Sydenham - Tempe - St Peters,354
21,Sydney (North) - Millers Point,315
11,Newtown (NSW),92
15,Potts Point - Woolloomooloo,73
12,Pagewood - Hillsdale - Daceyville,68
13,Petersham - Stanmore,64
2,Camperdown - Darlington,61
19,Surry Hills,61
6,Erskineville - Alexandria,59
7,Glebe - Forest Lodge,59


## Area 3

Area 3 is linked to `SA4_CODE = 118`, which corresponds to `Sydney - Eastern Suburbs`. Keeping the outputs separate makes it easier to show which SA2 and POI records belong to each selected area.

In [225]:
area_3_sa2_bbox_df, area_3_poi_df, area_3_poi_count_df = run_area_zone("area_3", "118")
display(area_3_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_3_poi_df.head())
display(area_3_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_3,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte
1,area_3,118,Sydney - Eastern Suburbs,118011340,Bondi Beach - North Bondi
2,area_3,118,Sydney - Eastern Suburbs,118011341,Bondi Junction - Waverly
3,area_3,118,Sydney - Eastern Suburbs,118011342,Centennial Park
4,area_3,118,Sydney - Eastern Suburbs,118011344,Dover Heights
5,area_3,118,Sydney - Eastern Suburbs,118011345,Paddington - Moore Park
6,area_3,118,Sydney - Eastern Suburbs,118011346,Rose Bay - Vaucluse - Watsons Bay
7,area_3,118,Sydney - Eastern Suburbs,118011347,Woollahra
8,area_3,118,Sydney - Eastern Suburbs,118011649,Bellevue Hill
9,area_3,118,Sydney - Eastern Suburbs,118011650,Double Bay - Darling Point


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,2296,MARKS PARK,Park,3,151.275150,-33.898499,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte,area_3
1,2297,WAVERLEY CEMETERY,Cemetery,1,151.266912,-33.908876,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte,area_3
2,27165,BET YOSEF (THE CARO SYNAGOGUE),Place Of Worship,1,151.261588,-33.888442,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte,area_3
3,27172,MARLBOROUGH RESERVE,Park,3,151.259434,-33.899592,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte,area_3
4,27173,BONDI PRESBYTERIAN CHURCH,Place Of Worship,1,151.267429,-33.893458,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte,area_3


,sa2_name,poi_count
18,Rose Bay - Vaucluse - Watsons Bay,141
10,Malabar - La Perouse,119
15,Paddington - Moore Park,85
3,Bondi Junction - Waverly,77
5,Coogee - Clovelly,66
2,Bondi Beach - North Bondi,62
17,Randwick - South,51
14,Matraville - Chifley,50
0,Bellevue Hill,46
16,Randwick - North,43


## Area 4

Area 4 is linked to `SA4_CODE = 125`, which corresponds to `Sydney - Parramatta`. This final section follows the same structure so that all four areas can be combined consistently later.

In [226]:
area_4_sa2_bbox_df, area_4_poi_df, area_4_poi_count_df = run_area_zone("area_4", "125")
display(area_4_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_4_poi_df.head())
display(area_4_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_4,125,Sydney - Parramatta,125011475,Rookwood Cemetery
1,area_4,125,Sydney - Parramatta,125011582,Auburn - Central
2,area_4,125,Sydney - Parramatta,125011583,Auburn - North
3,area_4,125,Sydney - Parramatta,125011584,Auburn - South
4,area_4,125,Sydney - Parramatta,125011585,Berala
5,area_4,125,Sydney - Parramatta,125011586,Lidcombe
6,area_4,125,Sydney - Parramatta,125011587,Regents Park
7,area_4,125,Sydney - Parramatta,125011709,Silverwater - Newington
8,area_4,125,Sydney - Parramatta,125011710,Wentworth Point - Sydney Olympic Park
9,area_4,125,Sydney - Parramatta,125021477,Ermington - Rydalmere


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,1149,ROOKWOOD CEMETERY,Cemetery,1,151.053588,-33.868984,125,Sydney - Parramatta,125011475,Rookwood Cemetery,area_4
1,2903,ROOKWOOD CEMETERY,Cemetery,1,151.054700,-33.876471,125,Sydney - Parramatta,125011475,Rookwood Cemetery,area_4
2,27012,ROOKWOOD MEMORIAL GARDENS AND CREMATORIUM,Crematorium,1,151.059782,-33.877265,125,Sydney - Parramatta,125011475,Rookwood Cemetery,area_4
3,53271,ROOKWOOD,Suburb,8,151.054542,-33.878671,125,Sydney - Parramatta,125011475,Rookwood Cemetery,area_4
4,133745,ROOKWOOD CEMETERY AND NECROPOLIS,Historic Site,3,151.051350,-33.869457,125,Sydney - Parramatta,125011475,Rookwood Cemetery,area_4


,sa2_name,poi_count
15,North Parramatta,132
7,Ermington - Rydalmere,119
14,Merrylands - Holroyd,112
9,Granville - Clyde,105
17,Northmead,97
12,Guildford West - Merrylands West,96
30,Wentworth Point - Sydney Olympic Park,95
19,Parramatta - North,94
11,Guildford - South Granville,84
13,Lidcombe,83


## Combine and save

After each area's SA2 and POI data has been collected, the four outputs are concatenated into two full tables: one SA2 table and one POI table. These tables are then saved into a local SQLite database. The database links `poi` to `sa2_bbox` through the `sa2_main` field, which makes it possible to run joined queries later.

In [227]:
all_sa2_bbox_df = pd.concat(
    [area_1_sa2_bbox_df, area_2_sa2_bbox_df, area_3_sa2_bbox_df, area_4_sa2_bbox_df],
    ignore_index=True,
)

all_poi_df = pd.concat(
    [area_1_poi_df, area_2_poi_df, area_3_poi_df, area_4_poi_df],
    ignore_index=True,
)

all_sa2_bbox_df = all_sa2_bbox_df[
    ["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name", "state_name", "area_sqkm", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
]
all_poi_df = all_poi_df[
    ["objectid", "poiname", "poitype", "poigroup", "longitude", "latitude", "area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]
]

db_path = current_dir / "data" / "task2_poi.db"
task2_sa2.save_to_sqlite(all_sa2_bbox_df, all_poi_df, str(db_path))

schema_df = task2_sa2.read_sql(
    "select name, sql from sqlite_master where type = 'table' order by name",
    str(db_path),
)
join_df = task2_sa2.read_sql(
    "select p.poi_id, p.area_name, p.poiname, p.poitype, s.sa4_code, s.sa4_name, s.sa2_name from poi p join sa2_bbox s on p.sa2_main = s.sa2_main limit 20",
    str(db_path),
)

display(schema_df)
display(join_df)


,name,sql
0,poi,"CREATE TABLE poi (\n poi_id integer primary key autoincrement,\n objectid integer,\n poiname text,\n poitype text,\n poigroup integer,\n longitude real,\n latitude real,\n area_name text,\n sa4_code text,\n sa4_name text,\n sa2_main text,\n sa2_name text,\n foreign key (sa2_main) references sa2_bbox(sa2_main)\n )"
1,sa2_bbox,"CREATE TABLE sa2_bbox (\n area_name text,\n sa4_code text,\n sa4_name text,\n sa2_main text primary key,\n sa2_name text,\n state_name text,\n area_sqkm real,\n bbox_xmin real,\n bbox_ymin real,\n bbox_xmax real,\n bbox_ymax real\n )"
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"


,poi_id,area_name,poiname,poitype,sa4_code,sa4_name,sa2_name
0,1,area_1,None,Place Of Worship,120,Sydney - Inner West,Concord - Mortlake - Cabarita
1,2,area_1,None,Place Of Worship,120,Sydney - Inner West,Concord - Mortlake - Cabarita
2,3,area_1,CONCORD GOLF COURSE,Golf Course,120,Sydney - Inner West,Concord - Mortlake - Cabarita
3,4,area_1,PRINCE EDWARD PARK,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
4,5,area_1,MASSEY PARK GOLF COURSE,Golf Course,120,Sydney - Inner West,Concord - Mortlake - Cabarita
5,6,area_1,STEWART RESERVE,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
6,7,area_1,HENLEY PARK,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
7,8,area_1,CENTRAL PARK,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
8,9,area_1,BAYVIEW PARK,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
9,10,area_1,WANGAL RESERVE,Park,120,Sydney - Inner West,Concord - Mortlake - Cabarita
